# 03 — Frequency-Domain Coupling (main η analysis)

The central analysis: ground-to-cable strain-transfer FRFs from Welch spectra,
with **four estimators** (direct, H1, H2, Hv), coherence gating, and the
η-vs-Θ summary across all configurations.

### Signals (full sweep record, displacement [m])

| Signal | Definition | Role |
|---|---|---|
| $X(t)=\delta L(t)$ | endpoint-pair span change $u_\parallel(\mathrm{right})-u_\parallel(\mathrm{left})$ (default; or shaker) | input |
| $Y_1(t)=\delta x_\ell(t)$ | Σ per-segment 3-D arc-length elongation | axial output → η |
| $Y_2(t)=u_{\perp,\mathrm{mid}}(t)$ | midpoint displacement ⊥ chord, signed along static sag | bending output |

### Estimators (Welch: nperseg = 2·fs → Δf = 0.5 Hz, 50 % overlap, Hann)

$$H_1=\frac{S_{XY}}{S_{XX}},\qquad H_2=\frac{S_{YY}}{S_{YX}},\qquad
H_v:\ \text{total least squares},\qquad
H_\mathrm{direct}=\frac{\mathcal F\{Y\}}{\mathcal F\{X\}}$$

$|H_1|\le|H_\mathrm{true}|\le|H_2|$ and $\gamma^2=|H_1|/|H_2|$ — the
spread between H1 and H2 widens exactly where coherence drops.

| § | Content |
|---|---|
| 1 | Selection, loading, geometry, signals |
| 2 | Transfer functions (all estimators) + resonance summary |
| 3 | Four-panel FRF figure (wrapped phase) |
| 4 | Estimator comparison H1 / H2 / Hv / direct |
| 5 | STFT-per-frequency cross-check |
| 6 | η vs Θ across ALL configurations (batch) |
| 7 | Export → `results/freqdomain_frf.npz`, `results/freqdomain_eta.npz` |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Editable reload while iterating on the package
%load_ext autoreload
%autoreload 2

import ldv_analysis as la
from ldv_analysis import config, io, analysis, plotting, export

plt.rcParams.update({
    "font.size": 11, "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "figure.dpi": 120, "savefig.dpi": 300, "figure.autolayout": True,
    "axes.titlesize": 11, "axes.labelsize": 10, "axes.linewidth": 0.8,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": "--",
    "lines.linewidth": 1.2,
})
print(f"{len(la.ALL_DATASETS)} datasets in catalogue.")

---
## § 1  Selection, loading, geometry, signals

In [ ]:
from ldv_analysis import freqdomain

# Overlay 2–4 datasets spanning a range of Θ for the FRF figures.
ACTIVE_LABELS = ["Cable6_10cm", "Cable6_10cm_Sag"]
# ACTIVE_LABELS = ["Cable5_5cm", "Cable5_10cm", "Cable5_15cm"]   # gap sweep

X_SOURCE = 'ends'      # 'ends' (recommended) | 'shaker' | 'left_sensor'
ESTIMATOR = 'h1'       # primary estimator for the η summaries

DATASETS = config.select_datasets(ACTIVE_LABELS)
for cfg in DATASETS:
    io.load_cable_dataset(cfg, verbose=False)
    analysis.prepare_geometry(cfg, sag_use_3d=True, sag_use_parabola=True)
    freqdomain.build_coupling_signals(cfg, x_source=X_SOURCE)
    print(f"  {cfg['label']:20s} Θ={cfg['theta_pred']:.4g}  "
          f"η_pred={cfg['eta_pred']:.3f}")

---
## § 2  Transfer functions & resonance summary

`compute_transfer_functions(..., all_estimators=True)` stores the primary
H_eta/H_mid (chosen `ESTIMATOR`) **and** every variant
(`H_eta_h1/h2/hv`, `H_eta_direct` on the dense FFT grid).

In [ ]:
RES_FMAX = 300.0

rows = []
for cfg in DATASETS:
    tf = freqdomain.compute_transfer_functions(cfg, estimator=ESTIMATOR,
                                               all_estimators=True)
    r = freqdomain.resonance_summary(cfg, tf, f_max=RES_FMAX)
    rows.append(dict(dataset=r['label'], Theta=round(cfg['theta_pred'], 4),
                     eta_pred=round(cfg['eta_pred'], 3),
                     f1_meas_Hz=round(r['f1_meas'], 1),
                     Q=round(r['Q'], 2) if np.isfinite(r['Q']) else None,
                     zeta=round(r['zeta'], 4) if np.isfinite(r['zeta']) else None,
                     df_Hz=tf['df'], nperseg=tf['nperseg']))
pd.DataFrame(rows)

---
## § 3  Four-panel FRF figure

Phase panels use the **wrapped** phase by default (`phase_mode='unwrapped'`
recovers the continuous Simone-Fig.-3 style, which can run away when
low-coherence bins inject spurious ±2π jumps).

In [ ]:
F_MAX_PLOT = 500.0

plotting.plot_fig3_transfer(DATASETS, f_max=F_MAX_PLOT, coh_thresh=0.7,
                            show_stft=False, show_eta_pred=True,
                            phase_mode='wrapped')

---
## § 4  Estimator comparison

The processing-chapter figure: direct (single FFT, no averaging) as the noisy
baseline, H1/H2 as the noise-on-output / noise-on-input least-squares fits,
Hv as the total-least-squares compromise, and the coherence that explains
where they diverge.

In [ ]:
plotting.plot_estimator_comparison(DATASETS[0], which='eta',
                                   f_max=300.0, coh_thresh=0.7)

---
## § 5  STFT-per-frequency cross-check

In [ ]:
STFT_FREQS = [5, 10, 20, 50, 100, 200, 300]

for cfg in DATASETS:
    freqdomain.stft_point_transfers(cfg, STFT_FREQS)

plotting.plot_fig3_transfer(DATASETS, f_max=F_MAX_PLOT, coh_thresh=0.7,
                            show_stft=True, show_eta_pred=True,
                            phase_mode='wrapped')

---
## § 6  η vs Θ across all configurations

Batch: for every sweep dataset compute the coherence-gated band mean
$\eta=\langle|H_\eta(f)|\rangle$ over the quasi-static band, then scatter
against Θ with the theory curve η = 1/(1+Θ).

> First run is heavy (~42 .mat loads); cached afterwards.

In [ ]:
ETA_LABELS = config.SWEEP_LABELS
ETA_DATASETS = config.select_datasets(ETA_LABELS)

ETA_FMIN, ETA_FMAX, ETA_COH = 5.0, 25.0, 0.7

for cfg in ETA_DATASETS:
    io.load_cable_dataset(cfg, verbose=False)
    analysis.prepare_geometry(cfg, sag_use_3d=True, sag_use_parabola=True)
    freqdomain.build_coupling_signals(cfg, x_source=X_SOURCE)
    freqdomain.compute_transfer_functions(cfg, estimator=ESTIMATOR,
                                          all_estimators=True)
    em, es, n = freqdomain.band_mean_eta(cfg, ETA_FMIN, ETA_FMAX, ETA_COH)
    print(f"  {cfg['label']:20s} Θ={cfg['theta_pred']:.4g}  "
          f"η_meas={('%.3f' % em) if em is not None else ' n/a':>6}  (n={n})")

In [ ]:
plotting.plot_eta_vs_theta_fd(
    ETA_DATASETS, f_min=ETA_FMIN, f_max=ETA_FMAX, coh_thresh=ETA_COH,
    theta_source='sag', x_source=X_SOURCE, percent=True)
# theta_source='material' uses Θ from ρ, E instead of the measured sag.

---
## § 7  Export

- `freqdomain_eta.npz` — one row per configuration: Θ (both sources),
  η_pred, coherence-gated band-mean η for **each estimator** (h1/h2/hv).
- `freqdomain_frf.npz` — full FRF curves (|H|, wrapped phase, γ²) for the
  datasets selected in § 1, for the paper FRF figure.

In [ ]:
def band_mean(tf, key, fmin, fmax, coh_thresh):
    f = tf['f']
    m = (f >= fmin) & (f <= fmax) & (tf['coh_eta'] >= coh_thresh)
    if not m.any():
        return np.nan, np.nan, 0
    v = np.abs(tf[key][m])
    return float(v.mean()), float(v.std()), int(m.sum())

acc = dict(labels=[], cable=[], gap_m=[], is_sag_variant=[],
           theta_sag=[], theta_material=[], theta_material_std=[],
           eta_pred=[], f1_meas=[],
           eta_h1=[], eta_h1_std=[], eta_h2=[], eta_h2_std=[],
           eta_hv=[], eta_hv_std=[], n_bins=[])
for cfg in ETA_DATASETS:
    tf = cfg['fd_tf']
    acc['labels'].append(cfg['label'])
    acc['cable'].append(cfg['cable'])
    acc['gap_m'].append(cfg['gap_m'])
    acc['is_sag_variant'].append('Sag' in cfg['label'])
    acc['theta_sag'].append(cfg['theta_pred'])
    acc['theta_material'].append(cfg['theta_material'])
    acc['theta_material_std'].append(cfg['theta_material_std'])
    acc['eta_pred'].append(cfg['eta_pred'])
    res = cfg.get('fd_resonance') or {}
    acc['f1_meas'].append(res.get('f1_meas', np.nan))
    for est in ('h1', 'h2', 'hv'):
        m, s, n = band_mean(tf, f'H_eta_{est}', ETA_FMIN, ETA_FMAX, ETA_COH)
        acc[f'eta_{est}'].append(m)
        acc[f'eta_{est}_std'].append(s)
    acc['n_bins'].append(n)

acc['is_sag_variant'] = np.array(acc['is_sag_variant'])
export.export_results(
    'freqdomain_eta',
    meta=dict(band=(ETA_FMIN, ETA_FMAX), coh_thresh=ETA_COH,
              x_source=X_SOURCE, estimator_primary=ESTIMATOR),
    **acc)

In [ ]:
# FRF curves of the § 1 selection (shared Welch grid).
tf0 = DATASETS[0]['fd_tf']
export.export_results(
    'freqdomain_frf',
    meta=dict(x_source=X_SOURCE, estimator=ESTIMATOR,
              nperseg=tf0['nperseg'], df=tf0['df'], coh_thresh=0.7),
    labels=[c['label'] for c in DATASETS],
    theta=[c['theta_pred'] for c in DATASETS],
    eta_pred=[c['eta_pred'] for c in DATASETS],
    f1_meas=[c['fd_resonance']['f1_meas'] for c in DATASETS],
    f=tf0['f'],
    H_eta_abs=[np.abs(c['fd_tf']['H_eta']) for c in DATASETS],
    H_eta_phase=[np.angle(c['fd_tf']['H_eta']) for c in DATASETS],
    coh_eta=[c['fd_tf']['coh_eta'] for c in DATASETS],
    H_mid_abs=[np.abs(c['fd_tf']['H_mid']) for c in DATASETS],
    H_mid_phase=[np.angle(c['fd_tf']['H_mid']) for c in DATASETS],
    coh_mid=[c['fd_tf']['coh_mid'] for c in DATASETS],
    H_eta_h2_abs=[np.abs(c['fd_tf']['H_eta_h2']) for c in DATASETS],
    H_eta_hv_abs=[np.abs(c['fd_tf']['H_eta_hv']) for c in DATASETS],
)